# 18j — six-scenario WIS bars by horizon

17j's ranking figure (`res/17j_scenarios_rank_pre_omicron.png`) sorts the series separately in each horizon
panel, so one series sits on a different row in every panel. This notebook draws the **same ranking** with the
rows fixed: horizons 1–4 as columns sharing y ticks.

Run 17j first. The bars and the dot plot read 17j's CSVs only. The age and over-time figures also read the four
forecast caches, because age-level WIS is in no CSV and 95% intervals need the draws. It reads those caches with
`load_9j_assembly`, so it cannot refit. It needs no CoMix or inc2prev read. The scores and coverage are then cached in
`dt_intermediate/18j_age_coverage_*`, so later runs skip the forecast caches and the R pass; set `REBUILD_18J = true` to redo them.

**Rows, top to bottom.** The two no-contact series, then four bars per contact model: contact (frozen) only,
contact (frozen) – variant, contact only, contact – variant.

**Contact-model order.** Each model is scored by its `variant + contact` mean WIS averaged over h1–4. Each level
of each factor (degree: hweibull/negbin; NGM: nbhd/mean/mean-diagonal) is ranked by the best score of any model
with that level, and models are sorted by (degree rank, NGM rank). The degree models therefore stay together, and
a model can sit above one that scores better, so read the scores from the bars and not from the row position.

**Window.** As in 17j's ranking: log-scale WIS over origins up to and including 2021-11-01, before Omicron.

**Second figure: Cleveland dot plot.** The same frame at horizons 1 and 4, as mean WIS relative to no contact – no
variant at that horizon (the line at 1): one row per contact model with its four scenarios, under a no-contact row
with its two (no variant, variant). Both columns share one x-axis. A second version shows all four horizons.

**Age and over-time figures**, at horizons 1 and 4 over the same pre-Omicron origins:
- **Relative WIS by age:** the series' mean WIS for each age group ÷ no contact – no variant's for that age and horizon.
- **WIS and coverage over time:** per-origin mean WIS, and the share of the 7 age groups whose observed infections
  fall inside the 95% interval, with the pandemic periods shaded. Four series are coloured and the rest grey.
- **Infections over time with forecasts:** the observed infections and, from every 4th origin, the h1–h4 median with
  its 50% and 95% bands for hweibull|nbhd with contact only and with contact – variant, on a log axis.
  One panel per age group (all ages first), 4 rows × 2 columns. A second version adds negbin|mean with contact – variant. That version is also drawn on the linear scale.

**Secondary attack rate** (last section): γ_SAR,v of the contact – variant models and no contact – variant over
the pre-Omicron origins at h = 1, one panel per model and one line per variant category, on a linear axis, with each category's genome share on
a right-hand axis. It reads
the `-variant` Stage-2 files directly (read-only).
The relative susceptibility and infectivity of the same models follow, as in 8j's ratio figures. A last figure merges the three for four of the models. A summary figure closes the notebook: the dot plot, 17j's coverage and bias, and 19r's Tweedie effects (run 19r first).

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")
include("9j_viz_utils.jl")
include("15j_viz_utils.jl")
include("17j_viz_utils.jl")
include("18j_viz_utils.jl")
using Statistics, DataFrames, CSV
default_plot_setting()
SAVE_DPI[] = parse(Int, get(ENV, "SAVE_DPI", "600"))   # high-resolution PNG export (save_show; display unchanged)

cfg = FrameworkConfig(constant_contacts = false)   # production defaults: only `horizons` is read here
@assert contacts_label(cfg) == "temporal-w8h-lc0-eng-nuts"
PRE_OMICRON_END = Date(2021, 11, 1)                # = 17j cell 24

In [ ]:
# 17j's ranking, and the check that it is what 17j's per-origin scores give for this window.
rk = CSV.read("../res/17j_scenarios_rank_pre_omicron.csv", DataFrame; stringtype = String)
n_pre = only(unique(rk.n_origins))
@assert nrow(rk) == 22 * length(cfg.horizons) && length(unique(rk.series)) == 22
@assert sort(unique(rk.horizon)) == collect(cfg.horizons) && only(unique(rk.scale)) == WIS_SCALE
@assert n_pre == 55
@assert nrow(unique(rk, [:model, :scenario, :horizon])) == nrow(rk)

let by_dt_h = rename(CSV.read("../res/17j_scenarios_scores_by_model_date_horizon.csv", DataFrame; stringtype = String),
                     :series => :model),
    st = scenario_table(filter(!=(NULL_LABEL), unique(rk.model))),
    re = scenario_rank_frame(scenario_wis_by_horizon_until(by_dt_h, PRE_OMICRON_END), st),
    a = sort(rk, [:series, :horizon]), b = sort(re, [:series, :horizon])
    @assert a.series == b.series && a.horizon == b.horizon && a.rank == b.rank && a.n_origins == b.n_origins
    @assert maximum(abs.(a.wis .- b.wis) ./ a.wis) < 1e-12
end
println("origins ≤ ", PRE_OMICRON_END, ": n = ", n_pre, " | ", length(unique(rk.series)), " series × ",
        length(cfg.horizons), " horizons — reproduced from the per-origin scores")

In [ ]:
# The contact-model order, what it was computed from, and the frame behind the figure.
ord = scenario_bar_contact_order(rk)
show(stdout, MIME("text/plain"), select(ord.models, :model, :mean_wis); allrows = true); println()
show(stdout, MIME("text/plain"), ord.levels; allrows = true); println()

rows = scenario_bar_rows(ord.order)
bar_frame = innerjoin(dropmissing(rows, :scenario), select(rk, :series, :model, :scenario, :horizon, :wis, :rank);
                      on = [:model, :scenario])
@assert nrow(dropmissing(rows, :scenario)) == 22 && nrow(bar_frame) == nrow(rk)   # every series drawn exactly once
sort!(bar_frame, [:horizon, order(:y; rev = true)])
CSV.write("../res/18j_scenarios_bars_by_horizon.csv", bar_frame)
show(stdout, MIME("text/plain"),
     unstack(select(bar_frame, :tick, :model, :y, :horizon, :wis), :horizon, :wis; renamecols = h -> "h$(h)");
     allrows = true, truncate = 40); println()

In [ ]:
save_show(plot_scenario_bars_by_horizon(rk, cfg; title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_scenarios_bars_by_horizon.png");

In [ ]:
# Cleveland dot plot of all 22 series at h1 and h4: mean WIS relative to no contact – no variant at that horizon,
# one row per contact model, under the no-contact row.
DOT_HORIZONS = (1, 4)
let d = scenario_dots_frame(rk; horizons = DOT_HORIZONS)
    @assert nrow(d) == 22 * length(DOT_HORIZONS)
    @assert all(==(1), @subset(d, :scenario .== "no contact").rel_wis)
    show(stdout, MIME("text/plain"), select(unstack(select(d, :model, :horizon, :scenario, :rel_wis), :scenario, :rel_wis),
                                            :model, :horizon, first.(BAR_NULL_ROWS)..., first.(BAR_CONTACT_ROWS)...);
         allrows = true, truncate = 40); println()
end
save_show(plot_scenario_dots(rk; horizons = DOT_HORIZONS, title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_scenarios_dots_h1_h4.png");

In [ ]:
# The same dot plot at all four horizons.
save_show(plot_scenario_dots(rk; horizons = (1, 2, 3, 4), panel_width = 380,
                             title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_scenarios_dots_h1_h2_h3_h4.png");

In [ ]:
# The age-level scores and coverage: from 18j's own cache, or built once from the four forecast caches. Those are
# read with `load_9j_assembly` and never `assemble_or_load_forecasts`, which fits on a miss.
REBUILD_18J = false
grid = cis_age_grid()
sources = scenario_assembly_sources(cfg)
scores_pre = scenario_age_coverage_or_load(cfg, sources, unique(rk.series), grid; horizons = DOT_HORIZONS,
                                           origin_max = PRE_OMICRON_END, rebuild = REBUILD_18J)
cov95, cov90 = scores_pre.cov95, scores_pre.cov90

In [ ]:
# Check the scores and the draw-based coverage against 17j's own per-origin scores, on the cached path too.
by_dt_h_csv = rename(CSV.read("../res/17j_scenarios_scores_by_model_date_horizon.csv", DataFrame; stringtype = String),
                     :series => :model)
let got = @subset(scores_pre.by_model_dt_h, :scale .== WIS_SCALE),
    want = @subset(by_dt_h_csv, :scale .== WIS_SCALE, :forecast_date .<= PRE_OMICRON_END, in.(:horizon, Ref(DOT_HORIZONS))),
    j = innerjoin(got, select(want, :model, :forecast_date, :horizon, :wis => :wis_csv); on = [:model, :forecast_date, :horizon]),
    c = innerjoin(select(got, :model => :series, :forecast_date, :horizon, :interval_coverage_90), cov90;
                  on = [:series, :forecast_date, :horizon])
    @assert nrow(got) == nrow(want) == nrow(j) == 22 * n_pre * length(DOT_HORIZONS)
    @assert maximum(abs.(j.wis .- j.wis_csv) ./ j.wis_csv) < 1e-9 "restricted scoring ≠ 17j's per-origin WIS"
    # The tripwire for a transposed age or horizon index: the draw-based coverage at 90% = scoringutils' own.
    @assert nrow(c) == nrow(got) && maximum(abs.(c.interval_coverage_90 .- c.coverage)) < 1e-12 "draw-based 90% coverage ≠ scoringutils"
end
println("checked: WIS against 17j's CSV and draw-based 90% coverage against scoringutils, over ",
        22 * n_pre * length(DOT_HORIZONS), " series × origin × horizon")

In [ ]:
# Relative WIS by age: y = age group, x = the 22 series in the order of the bars, one row per horizon.
age_rel = scenario_age_rel_wis(scores_pre.by_model_h_age)
@assert nrow(age_rel) == 22 * grid.N * length(DOT_HORIZONS) && Set(age_rel.age_group) == Set(grid.LAB)
@assert all(==(1), @subset(age_rel, :series .== NULL_LABEL).rel_wis)
CSV.write("../res/18j_age_rel_wis_h1_h4.csv", sort(age_rel, [:horizon, :series, :age_group]))
println("relative WIS: ", join(["h$(h) $(join(round.(extrema(@subset(age_rel, :horizon .== h).rel_wis); digits = 2), "–"))" for h in DOT_HORIZONS], ", "))
# Colours capped at 1 ± 0.5: mean-diagonal's values reach 2.24 and would wash out the rest. Every cell prints its value.
save_show(plot_scenario_age_heatmap(age_rel, rk, grid; horizons = DOT_HORIZONS, cap = 0.5,
                                    title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_age_rel_wis_h1_h4.png");

In [ ]:
# Mean WIS (left) and 95% coverage (right) over time, with the pandemic periods shaded, one row per horizon.
@assert nrow(cov95) == 22 * n_pre * length(DOT_HORIZONS) && length(unique(cov95.series)) == 22
CSV.write("../res/18j_coverage95_by_origin_h1_h4.csv", cov95)
println("mean 95% coverage over the window: ", join(["$(s) h$(h) $(round(mean(@subset(cov95, :series .== s, :horizon .== h).coverage); digits = 3))"
                                                     for (s, _, _) in OVER_TIME_HIGHLIGHT for h in DOT_HORIZONS], ", "))
wis_time = @subset(rename(by_dt_h_csv, :model => :series), :forecast_date .<= PRE_OMICRON_END)
save_show(plot_scenario_wis_coverage_over_time(wis_time, cov95, grid; horizons = DOT_HORIZONS, nominal = 0.95,
                                               title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_wis_coverage_over_time_h1_h4.png");

In [ ]:
# Observed infections with the h1–h4 forecasts of hweibull|nbhd (contact only, contact – variant) from every 4th
# pre-Omicron origin: median and 50% / 95% bands, one panel per age group. The quantiles are cached like cell 7's scores,
# for the next figure's three series at once (a superset of this one's), so the two figures share one cache.
FAN_EVERY = 4
@assert issubset(first.(FAN_SERIES), first.(FAN_SERIES_NEGBIN))
fans_pre = scenario_fan_quantiles_or_load(cfg, sources, first.(FAN_SERIES_NEGBIN), grid; origin_max = PRE_OMICRON_END,
                                          rebuild = REBUILD_18J)
let q = fans_pre.quant, o = fans_pre.observed, H = length(cfg.horizons)
    @assert nrow(q) == length(FAN_SERIES_NEGBIN) * n_pre * H * (grid.N + 1)
    @assert all(q[!, "q0.025"] .<= q[!, "q0.25"] .<= q[!, "q0.5"] .<= q[!, "q0.75"] .<= q[!, "q0.975"])
    tot = combine(groupby(@subset(o, :age_group .!= "all"), :week), :observed => sum => :sum_ages)
    j = innerjoin(@subset(o, :age_group .== "all"), tot; on = :week)
    @assert nrow(j) == nrow(tot) && j.observed == j.sum_ages "all-ages observed ≠ Σ ages"
    # The tripwire for a transposed age or horizon index: the share of the 7 ages inside [q0.025, q0.975] = cell 7's cov95.
    qa = @transform(@subset(q, :age_group .!= "all", in.(:horizon, Ref(DOT_HORIZONS))),
                    :week = :forecast_date .+ Week.(:horizon))
    c = combine(groupby(innerjoin(qa, o; on = [:week, :age_group]), [:series, :forecast_date, :horizon]),
                ["observed", "q0.025", "q0.975"] => ((y, lo, hi) -> mean(lo .<= y .<= hi)) => :share)
    c = innerjoin(c, cov95; on = [:series, :forecast_date, :horizon])
    @assert nrow(c) == length(FAN_SERIES_NEGBIN) * n_pre * length(DOT_HORIZONS) && c.share == c.coverage "fan quantiles ≠ cell 7's 95% coverage"
end
fan_origins = sort(unique(fans_pre.quant.forecast_date))[1:FAN_EVERY:end]
println("drawn origins (", length(fan_origins), "): ", join(fan_origins, ", "))
CSV.write("../res/18j_fans_over_time_pre_omicron.csv",   # all three series; the negbin figure reads the same rows
          @subset(fans_pre.quant, in.(:forecast_date, Ref(fan_origins))))
save_show(plot_scenario_fans_over_time(fans_pre, grid; every = FAN_EVERY,
                                       title_suffix = ", origins ≤ $(PRE_OMICRON_END): every $(FAN_EVERY)th of $(n_pre)"),
          "../res/18j_fans_over_time_pre_omicron.png");

In [ ]:
# The same figure with negbin|mean (contact – variant) added, from the same cache and origins.
save_show(plot_scenario_fans_over_time(fans_pre, grid; series = FAN_SERIES_NEGBIN, every = FAN_EVERY,
                                       title_suffix = ", origins ≤ $(PRE_OMICRON_END): every $(FAN_EVERY)th of $(n_pre)"),
          "../res/18j_fans_over_time_negbin_pre_omicron.png");

In [ ]:
# The negbin figure on the raw (linear) scale. The y-axis is capped at 1.8× each panel's observed maximum, and the bands
# it cuts are counted in the panel titles.
save_show(plot_scenario_fans_over_time(fans_pre, grid; series = FAN_SERIES_NEGBIN, every = FAN_EVERY, yscale = :identity, ycap = 1.8,
                                       legend_pos = (0.42, 0.62),   # spring 2021, below the period labels: the bottom right holds autumn 2021 here
                                       title_suffix = ", origins ≤ $(PRE_OMICRON_END): every $(FAN_EVERY)th of $(n_pre)"),
          "../res/18j_fans_over_time_negbin_pre_omicron_linear.png");

## Secondary attack rate of the variant models over time

The per-contact secondary attack rate γ_SAR,v of the **contact – variant** models (contact conditional on the t₀+h
contacts), plus **no contact – variant** as a reference; mean-diagonal is dropped. The variant renewal fits one γ per
variant category chosen at the origin (9j §V), so each model gets a panel with one line per category: median and 90%
band at h = 1, pre-Omicron origins. A line breaks where its category has no renewal of its own.

The left axis (γ, solid) is linear and fixed at 0–1, except for hweibull|mean, whose γ runs past 1, and negbin|nbhd,
whose γ ≈ 0.005 would sit on the floor; those two get their own scale. Even the fixed panels are not comparable with
one another: γ_SAR multiplies C*, whose units differ by degree model and NGM builder. Read the levels over time and
across variants within a panel. Bands cut by the axis are counted in the panel title. The right axis (dashed, %) is
each category's share of the origin week's COG-UK genomes; a pooled category such as the residual shows its pooled
share. Read from the Stage-2 files with 9j's
`collect_variant_gamma`, which cannot fit.

In [ ]:
# γ_SAR,v per model × origin × variant (h = 1), read from the `-variant` Stage-2 files.
VG_DROP = "unweighted-negbin|mean-diagonal"
vg_models = vcat(filter(!=(VG_DROP), ord.order), [NULL_LABEL])
@assert VG_DROP in ord.order && length(vg_models) == 5
vg_origins = sort(unique(@subset(by_dt_h_csv, :forecast_date .<= PRE_OMICRON_END).forecast_date))
@assert length(vg_origins) == n_pre
vg = collect_variant_gamma(variant_gamma_combos(vg_models), [(; origin = o) for o in vg_origins], cfg; h = 1)
@assert nrow(unique(vg, [:model, :origin])) == length(vg_models) * n_pre "a variant Stage-2 file is missing"
@assert all(isfinite, vg.gamma_med) && all(>(0), vg.gamma_lo)
# The tripwire for a transposed variant index: one cell recomputed straight from its pooled file.
let r = vg[findfirst(r -> r.variant == "Alpha", eachrow(vg)), :],
    pooled = load(stage2_path(variant_gamma_combos([replace(r.model, "-variant" => "")])[1]..., r.origin, 1;
                              contacts = contacts_label(cfg), save_dir = "../dt_intermediate"), "pooled")
    @assert median(pooled.gamma_sar[:, findfirst(==("Alpha"), pooled.variants)]) == r.gamma_med
end
CSV.write("../res/18j_variant_gamma_pre_omicron.csv", vg)
show(stdout, MIME("text/plain"),
     combine(groupby(vg, [:model, :variant]), nrow => :n_origins, :gamma_med => median => :median_gamma,
             :frac_at_bound => (x -> count(>(0.05), x)) => :n_at_bound); allrows = true); println()
save_show(plot_variant_gamma_over_time(vg, vg_models; title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_variant_gamma_pre_omicron.png");

## Relative susceptibility and infectivity of the variant models

The same five models, origins and horizon as the γ_SAR figure, redrawn like 8j's
`res/8j_susc_ratio_over_time.png`: per age super-group, the ratio to the model's reference bin 25-34 (dashed line at 1),
median and 90% band (no prior band). Population-weighted super-groups 2-15 / 16-24 / 35-49 / 50+.
- The variant renewal SHARES susc and inf across variants (only γ_SAR is per variant), so there is one profile per fit.
- **Log** axis (a halving and a doubling sit equally far from 1), one per figure, set from the medians, so a wide band
  can run off the panel (counted in the cell output). Ticks are powers of 2 labelled as plain ratios.
- Read from the `-variant` Stage-2 files alone with `collect_variant_susc_inf`, the same arithmetic as 9j's
  `collect_transmission_structure` (which cannot read a variant file); nothing can fit.

In [ ]:
tr_v = collect_variant_susc_inf(vg_models, vg_origins, cfg; grid = grid)
@assert nrow(tr_v.frame) == length(vg_models) * n_pre * 2 * length(tr_v.sg_names)
@assert all(isfinite, tr_v.frame.ratio_med)
CSV.write("../res/18j_variant_susc_inf_ratio_pre_omicron.csv", tr_v.frame)
for (q, st, fname) in (("susceptibility", tr_v.susc, "susc"), ("infectivity", tr_v.inf, "infectivity"))
    # Log axis (a halving and a doubling sit equally far from 1), one per figure, limits from the medians
    # (9j's `_median_ylims`); ticks at powers of 2 labelled as plain ratios rather than 10^x.
    yl = _median_ylims(vcat([vec(st[l].med) for l in tr_v.labels]...); log = true)
    println(q, ": y-axis ", round.(yl; digits = 3), ", 90% band ends outside it: ",
            count(>(yl[2]), vcat([vec(st[l].hi) for l in tr_v.labels]...)) +
            count(<(yl[1]), vcat([vec(st[l].lo) for l in tr_v.labels]...)))
    save_show(plot_ratio(st, tr_v.labels, vg_origins,
                         "18j — $(q) ratio to $(tr_v.ref_lab), variant models (h = 1, log axis), origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))";
                         gnames = tr_v.sg_names, ref_lab = tr_v.ref_lab, ylims = yl,   # no `cfg` ⇒ no prior band
                         yticks = _pow2_ticks(yl), marker_kw = (; msw = 0)),
              "../res/18j_variant_$(fname)_ratio_pre_omicron.png");
end

## One figure: γ_SAR, susceptibility and infectivity

The three figures above merged for four models (columns: hweibull|nbhd, hweibull|mean, negbin|mean and no contact, all
with the variant renewal; rows: γ_SAR,v, susceptibility, infectivity). Each row keeps its figure's axes: γ linear and
fixed at 0–1 except hweibull|mean, with the variant share in % on the right; susceptibility and infectivity on a log axis
with plain-ratio ticks, one range per row from the medians. The data are in the γ and susc/inf CSVs above.

In [ ]:
TG_MODELS = ["weighted-hweibull|neighbourhood", "weighted-hweibull|mean", "unweighted-negbin|mean", NULL_LABEL]
@assert issubset(TG_MODELS, vg_models)
save_show(plot_variant_transmission_grid(vg, tr_v, TG_MODELS, vg_origins;
                                         title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_variant_transmission_pre_omicron.png"; l = 16Plots.mm);   # the "0.125" ticks need the room

## Summary figure: relative WIS, coverage and bias, and the regression effects

One pre-Omicron figure (origins ≤ 2021-11-01, n = 55):
- **left, top:** the dot plot above at horizons 1 and 4 (`18j_scenarios_dots_h1_h4.png`);
- **left, bottom:** 50% / 90% coverage and bias of the contact – variant models against no contact and no contact +
  variant (17j's `17j_variant_coverage_bias_pre_omicron.png`, recomputed here from 17j's per-origin CSV);
- **right:** 19r's Tweedie fixed effects (`19r_fixed_effects_tweedie.png`) redrawn with Plots from
  `res/19r_fixed_effects.csv`, with the derived block in the middle and cut to the two future-vs-frozen contrasts.
  Rows are sorted within each block by the h4 effect, largest WIS reduction at the top, as in 19r.

**Run 19r first** as well as 17j: this cell reads 19r's CSV.

In [ ]:
SUMMARY_MODELS = ["weighted-hweibull|neighbourhood", "weighted-hweibull|mean",
                  "unweighted-negbin|neighbourhood", "unweighted-negbin|mean"]
wis_pre_18 = scenario_wis_by_horizon_until(by_dt_h_csv, PRE_OMICRON_END)
@assert all(==(n_pre), wis_pre_18.n_origins)
eff_19r = CSV.read("../res/19r_fixed_effects.csv", DataFrame; stringtype = String)
let f = tweedie_forest_frame(eff_19r)
    @assert nrow(f) == sum(length(last(b)) for b in FOREST_BLOCKS) * length(unique(f.horizon))
    show(stdout, MIME("text/plain"), unstack(select(f, :block, :term, :horizon, :ratio_median), :horizon, :ratio_median;
                                             renamecols = h -> "h$(h)"); allrows = true); println()
end
save_show(plot_summary_pre_omicron(rk, wis_pre_18, eff_19r, cfg; models = SUMMARY_MODELS,
                                   title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_summary_pre_omicron.png"; l = 12Plots.mm);   # the 50% coverage panel's y-label needs the room